# Entrenamiento robusto de EPP con YOLO26 (CCTV/RTSP)

Este notebook reemplaza el script monolítico anterior. Sus objetivos son:

- evitar fuga temporal entre *frames* vecinos de una misma cámara;
- mantener intacto el dataset fuente en Google Drive;
- entrenar con degradaciones CCTV/RTSP moderadas y realistas;
- medir por separado el desempeño limpio y el desempeño degradado;
- seleccionar checkpoints con la cabeza end-to-end objetivo de producción;
- dejar QAT INT8 como una segunda fase opcional, no mezclarlo con el baseline.

**Importante:** no se intenta “forzar” confidencias altas. Una confianza alta y
equivocada es peor que una predicción prudente. Para seguridad industrial se
mejora la evidencia (datos, etiquetas, resolución y clases), se calibra por clase
y se usa persistencia temporal/tracking en producción.



### Descarga de un dataset en Kaggle
Descargué un dataset para mejorar el mío y luego mezclarlos.

In [ ]:
# ============================================================
# 0. LIBRERÍAS
# ============================================================
import os
import json
import random
import shutil
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from tqdm import tqdm
from kaggle.api.kaggle_api_extended import KaggleApi

# ============================================================
# 1. CONFIGURACIÓN Y RUTAS
# ============================================================
DATASET_HANDLE = "mugheesahmad/sh17-dataset-for-ppe-detection"
SEED = 42

BASE_DIR = Path("/content/sh17_procesado")
AUTO_DIR = BASE_DIR / "sh17_alineado"
REVIEW_DIR = BASE_DIR / "sh17_revision"

AUTO_IMAGES = AUTO_DIR / "images"
AUTO_LABELS = AUTO_DIR / "labels"
REVIEW_IMAGES = REVIEW_DIR / "images"
REVIEW_LABELS = REVIEW_DIR / "labels"

CACHE_DIR = Path("/content/sh17_cache")
LABEL_CACHE = CACHE_DIR / "labels"

for d in [AUTO_IMAGES, AUTO_LABELS, REVIEW_IMAGES, REVIEW_LABELS, LABEL_CACHE]:
    d.mkdir(parents=True, exist_ok=True)

rng = random.Random(SEED)

LOCAL_NAMES = {
    0: "Gloves", 1: "Person", 2: "Safety_boots", 3: "Vest",
    4: "respirador", 5: "tapaorejas", 6: "Hard_hat", 7: "lentes_protectores",
}

SH17_NAMES = {
    0: "person", 1: "ear", 2: "ear-mufs", 3: "face", 4: "face-guard",
    5: "face-mask", 6: "foot", 7: "tool", 8: "glasses", 9: "gloves",
    10: "helmet", 11: "hands", 12: "head", 13: "medical-suit",
    14: "shoes", 15: "safety-suit", 16: "safety-vest",
}
SH17_NAME_TO_ID = {name: cid for cid, name in SH17_NAMES.items()}

AUTO_MAPPING = {9: 0, 0: 1, 16: 3, 10: 6}
REVIEW_MAPPING = {**AUTO_MAPPING, 14: 2, 2: 5, 8: 7}

AUTO_QUOTAS = {"gloves": 600, "helmet": 500, "safety-vest": 400}
REVIEW_QUOTAS = {"ear-mufs": 300, "glasses": 400, "shoes": 400}

AUTO_TARGET_IDS = {SH17_NAME_TO_ID[k]: v for k, v in AUTO_QUOTAS.items()}
REVIEW_TARGET_IDS = {SH17_NAME_TO_ID[k]: v for k, v in REVIEW_QUOTAS.items()}
BLOCKED_FOR_AUTOMATIC = {2, 5, 8, 14}

# ============================================================
# 2. AUTENTICACIÓN Y SESIÓN DIRECTA HTTP DE KAGGLE (CORREGIDO)
# ============================================================
api = KaggleApi()
api.authenticate()

# Extraer usuario y API token de forma robusta
kaggle_username = getattr(api, "username", None) or os.getenv("KAGGLE_USERNAME")
kaggle_key = getattr(api, "key", None) or os.getenv("KAGGLE_KEY")

# Si no están directos, leerlos del archivo kaggle.json
if not (kaggle_username and kaggle_key):
    kaggle_json_path = Path.home() / ".kaggle" / "kaggle.json"
    if kaggle_json_path.is_file():
        credentials = json.loads(kaggle_json_path.read_text(encoding="utf-8"))
        kaggle_username = credentials.get("username")
        kaggle_key = credentials.get("key")

# Alternativa interna del cliente API de Kaggle
if not (kaggle_username and kaggle_key) and hasattr(api, "api_client"):
    kaggle_username = api.api_client.configuration.username
    kaggle_key = api.api_client.configuration.password

if not (kaggle_username and kaggle_key):
    raise ValueError(
        "No se pudieron resolver las credenciales de Kaggle. "
        "Asegúrate de tener KAGGLE_USERNAME y KAGGLE_KEY configurados o ~/.kaggle/kaggle.json presente."
    )

auth = (kaggle_username, kaggle_key)

# ============================================================
# 3. LISTAR ARCHIVOS REMOTOS
# ============================================================
print("Listando archivos remotos...")
remote_files = []
page_token = None

while True:
    res = api.dataset_list_files(DATASET_HANDLE, page_token=page_token, page_size=1000)
    remote_files.extend([f.name for f in res.files])
    page_token = getattr(res, 'next_page_token', None)
    if not page_token:
        break

print(f"Archivos remotos totales: {len(remote_files)}")

# Mapear imágenes y etiquetas por stem
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
remote_images = {Path(f).stem: f for f in remote_files if Path(f).suffix.lower() in IMG_EXTS}
remote_labels = {Path(f).stem: f for f in remote_files if Path(f).suffix.lower() == ".txt"}

# Conservar solo los que tengan par exacto
paired_stems = sorted(set(remote_images.keys()) & set(remote_labels.keys()))
print(f"Pares imagen-etiqueta encontrados: {len(paired_stems)}")

# ============================================================
# 4. DESCARGA RÁPIDA DE ETIQUETAS (HTTP DIRECTO)
# ============================================================
def download_file_http(remote_path, dest_path):
    if dest_path.is_file():
        return dest_path
    
    url = f"https://www.kaggle.com/api/v1/datasets/download/file/{DATASET_HANDLE}/{remote_path}"
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    
    resp = requests.get(url, auth=auth, stream=True, timeout=30)
    if resp.status_code == 200:
        with open(dest_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=32768):
                f.write(chunk)
        return dest_path
    raise RuntimeError(f"Error {resp.status_code} en {remote_path}")

print("\nDescargando etiquetas (.txt)...")
label_tasks = [(remote_labels[stem], LABEL_CACHE / f"{stem}.txt") for stem in paired_stems]

with ThreadPoolExecutor(max_workers=16) as executor:
    futures = [executor.submit(download_file_http, r, d) for r, d in label_tasks]
    for _ in tqdm(as_completed(futures), total=len(futures), desc="Descarga TXT"):
        pass

# ============================================================
# 5. PARSEAR ETIQUETAS LOCALES
# ============================================================
label_info = {}
candidates = defaultdict(list)

for stem in paired_stems:
    lbl_file = LABEL_CACHE / f"{stem}.txt"
    if not lbl_file.exists():
        continue
    
    lines = []
    class_ids = set()
    for line in lbl_file.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            try:
                cid = int(float(parts[0]))
                lines.append(parts)
                class_ids.add(cid)
            except ValueError:
                continue
    label_info[stem] = {"lines": lines, "class_ids": class_ids}
    for cid in class_ids:
        candidates[cid].append(stem)

# ============================================================
# 6. SELECCIÓN BALANCEADA
# ============================================================
def select_balanced(target_ids, blocked_ids=None):
    blocked_ids = set(blocked_ids or [])
    selected = set()
    ordered_classes = sorted(target_ids, key=lambda cid: len(candidates[cid]))

    for cid in ordered_classes:
        quota = target_ids[cid]
        eligible = [s for s in candidates[cid] if not (label_info[s]["class_ids"] & blocked_ids)]
        rng.shuffle(eligible)

        already_covered = sum(cid in label_info[s]["class_ids"] for s in selected)
        needed = len(eligible) if quota is None else max(0, quota - already_covered)
        new_items = [s for s in eligible if s not in selected][:needed]
        selected.update(new_items)

    return sorted(selected)

auto_stems = select_balanced(AUTO_TARGET_IDS, blocked_ids=BLOCKED_FOR_AUTOMATIC)
review_stems = select_balanced(REVIEW_TARGET_IDS, blocked_ids=None)
review_stems = [s for s in review_stems if s not in set(auto_stems)]

print(f"\nSeleccionadas automáticas: {len(auto_stems)}")
print(f"Seleccionadas revisión:    {len(review_stems)}")

# ============================================================
# 7. DESCARGAR SOLO LAS IMÁGENES SELECCIONADAS Y GUARDAR TXT
# ============================================================
def process_subset(stems, mapping, img_dir, lbl_dir, status):
    manifest = []
    
    def fetch_and_save(stem):
        # 1. Transformar etiquetas
        lines = label_info[stem]["lines"]
        remapped_lines = []
        for p in lines:
            cid = int(float(p[0]))
            if cid in mapping:
                remapped = p.copy()
                remapped[0] = str(mapping[cid])
                remapped_lines.append(" ".join(remapped))

        if not remapped_lines:
            return None

        # 2. Descargar imagen seleccionada únicamente
        remote_img_path = remote_images[stem]
        ext = Path(remote_img_path).suffix
        out_img_name = f"sh17__{stem}{ext}"
        out_lbl_name = f"sh17__{stem}.txt"
        
        target_img_path = img_dir / out_img_name
        download_file_http(remote_img_path, target_img_path)
        
        # 3. Escribir txt con IDs remapeados
        (lbl_dir / out_lbl_name).write_text("\n".join(remapped_lines) + "\n", encoding="utf-8")
        
        return {
            "status": status,
            "image": out_img_name,
            "label": out_lbl_name,
            "sh17_classes": list(label_info[stem]["class_ids"]),
            "remapped_classes": [int(l.split()[0]) for l in remapped_lines]
        }

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = [executor.submit(fetch_and_save, s) for s in stems]
        for f in tqdm(as_completed(futures), total=len(futures), desc=f"Descargando {status}"):
            res = f.result()
            if res:
                manifest.append(res)
                
    return manifest

print("\nDescargando únicamente imágenes seleccionadas...")
auto_manifest = process_subset(auto_stems, AUTO_MAPPING, AUTO_IMAGES, AUTO_LABELS, "automatico")
review_manifest = process_subset(review_stems, REVIEW_MAPPING, REVIEW_IMAGES, REVIEW_LABELS, "revision")

# Guardar manifiestos
pd.DataFrame(auto_manifest).to_csv(AUTO_DIR / "manifest.csv", index=False)
pd.DataFrame(review_manifest).to_csv(REVIEW_DIR / "manifest.csv", index=False)

# Guardar classes.txt
classes_text = "\n".join(LOCAL_NAMES[i] for i in sorted(LOCAL_NAMES)) + "\n"
(AUTO_DIR / "classes.txt").write_text(classes_text, encoding="utf-8")
(REVIEW_DIR / "classes.txt").write_text(classes_text, encoding="utf-8")

# Limpiar cache de etiquetas sobrantes
shutil.rmtree(CACHE_DIR)

print(f"\nFinalizado exitosamente:")
print(f" - Automático: {len(auto_manifest)} imágenes en {AUTO_IMAGES}")
print(f" - Revisión:   {len(review_manifest)} imágenes en {REVIEW_IMAGES}")

## 1. Instalación

Se instalan versiones actuales y luego se registra el entorno exacto dentro del
run. Para repetir un experimento meses después, usa ese `environment.txt`.



In [ ]:
%pip install -q -U ultralytics albumentations pyyaml pandas matplotlib onnx scikit-learn



In [ ]:
from google.colab import drive

drive.mount("/content/drive")



In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
from collections import defaultdict
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Iterable

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnx
import pandas as pd
import torch
import ultralytics
import yaml
from ultralytics import YOLO
from sklearn.model_selection import GroupKFold

print("Python:", sys.version.split()[0])
print("Ultralytics:", ultralytics.__version__)
print("Albumentations:", A.__version__)
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    raise RuntimeError("Activa un runtime con GPU en Colab antes de entrenar.")



## 2. Configuración del experimento

`PRODUCTION_NMS=False` fija la ruta objetivo eficiente: cabeza one-to-one
end-to-end `[1,300,6]`, sin supresión IoU. La rama raw es `[1,12,8400]` a
640 px y `[1,12,18900]` a 960 px; se conserva con NMS únicamente como control
A/B para cuantificar la diferencia de mAP.

La validación cruzada agrupa secuencias por cámara/video usando el
nombre o la carpeta. Si conoces el patrón exacto, define `GROUP_REGEX` y captura
un grupo llamado `group`, por ejemplo:

```python
GROUP_REGEX = r"(?P<group>cam\d+_2026-\d{2}-\d{2})"
```



In [ ]:
PPE_CLASS_NAMES = (
    "Gloves",
    "Person",
    "Safety_boots",
    "Vest",
    "respirador",
    "tapaorejas",
    "Hard_hat",
    "lentes_protectores",
)

DATASET_DIR = Path("/content/drive/MyDrive/EPP_monitor/dataset-08-09")
BASE_MODEL = "yolo26n.pt"

SEED = 42
K_FOLDS = 5  # Ajusta a 2–5 si hay pocas secuencias independientes por clase.
GROUP_REGEX: str | None = None

WORK_ROOT = Path("/content/epp_yolo26_cctv_v2")
SPLIT_ROOT = WORK_ROOT / f"dataset_groupkfold{K_FOLDS}"
RUNS_ROOT = Path("/content/runs/ppe")
DRIVE_RUNS_ROOT = Path("/content/drive/MyDrive/EPP_monitor/runs")
OUTPUT_MODEL = Path("/content/drive/MyDrive/EPP_monitor/best_ppe_yolo26n_cctv_v2.pt")
BENCHMARK_ROOT = Path(
    "/content/drive/MyDrive/EPP_monitor/benchmarks/"
    f"EPP_val_{DATASET_DIR.name}_cctv_v2_groupkfold{K_FOLDS}"
)

EPOCHS = 140
PATIENCE = 25
IMGSZ = 960
BATCH = 0.75            # AutoBatch: utiliza aproximadamente 75 % de la VRAM.
VAL_BATCH = 8           # Lote fijo: validación no usa AutoBatch fraccional.
DEVICE = 0
WORKERS = min(4, max(2, (os.cpu_count() or 2) // 2))
CACHE_MODE: str | bool = "auto"  # RAM si cabe con margen; de lo contrario disco local.
FAST_TRAINING = True     # Desactiva determinismo estricto y habilita autotuning cuDNN.
COMPILE_MODE: str | bool = "reduce-overhead"

PRODUCTION_NMS = False
COMPARE_BOTH_HEADS = True
RUN_ROBUSTNESS_BENCHMARK = True
EXPORT_END2END_ONNX = True

# Desbalance: 0 desactiva; 1 aplica el inverso completo. Empezar conservador.
CLS_PW = 0.25

# Degradaciones in-memory. Probabilidades bajas para no convertir el dataset en
# imágenes irreales. El motion blur vuelve de forma controlada (6 %), no masiva.
MOTION_BLUR_P = 0.06
ATMOSPHERIC_P = 0.04
OCCLUSION_P = 0.03

# QAT es una fase separada. Activar solo si PTQ INT8 no cumple el criterio.
RUN_QAT_INT8 = False
QAT_EPOCHS = 20

RUN_NAME = f"ppe_yolo26n_cctv_v2_i{IMGSZ}_k{K_FOLDS}_seed{SEED}"

for path in (WORK_ROOT, RUNS_ROOT, DRIVE_RUNS_ROOT, OUTPUT_MODEL.parent, BENCHMARK_ROOT.parent):
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)



## 3. Auditoría de etiquetas y K-fold sin fuga temporal

Si el dataset ya tiene `train` y `val`, ambos forman el conjunto de
validación cruzada; `test` queda reservado fuera de los folds. Cada secuencia
completa se asigna a un solo fold y se valida exactamente una vez. Las
carpetas internas de `images/` y `labels/` pueden ser diferentes: primero se
intenta la ruta espejo y después una coincidencia única por nombre de archivo.
Las copias con imagen y anotación equivalentes se excluyen del conjunto;
si una misma imagen tiene anotaciones distintas, se detiene para revisión.
Si no puede inferirse una secuencia compartida, se avisa para que definas
`GROUP_REGEX`; nunca se vuelve a una división aleatoria única por imagen.



In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


@dataclass(frozen=True)
class Sample:
    image: Path
    label: Path
    relative: Path
    counts: tuple[int, ...]
    group: str


def normalize_names(names: list | dict) -> tuple[str, ...]:
    if isinstance(names, dict):
        normalized = tuple(str(names.get(i, names.get(str(i)))) for i in range(len(names)))
    else:
        normalized = tuple(str(x) for x in names)
    if normalized != PPE_CLASS_NAMES:
        raise ValueError(
            "El orden de clases no coincide con producción.\n"
            f"Esperado: {list(PPE_CLASS_NAMES)}\nRecibido: {list(normalized)}"
        )
    return normalized


def load_dataset_names(dataset_dir: Path) -> tuple[str, ...]:
    data_yaml = dataset_dir / "data.yaml"
    classes_txt = dataset_dir / "classes.txt"
    if data_yaml.is_file():
        data = yaml.safe_load(data_yaml.read_text(encoding="utf-8")) or {}
        if "names" in data:
            return normalize_names(data["names"])
    if classes_txt.is_file():
        names = [x.strip() for x in classes_txt.read_text(encoding="utf-8-sig").splitlines() if x.strip()]
        return normalize_names(names)
    raise FileNotFoundError("No se encontró data.yaml ni classes.txt con el contrato de clases.")


def parse_label(path: Path) -> tuple[int, ...]:
    counts = [0] * len(PPE_CLASS_NAMES)
    for line_number, raw in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), start=1):
        line = raw.strip()
        if not line:
            continue
        fields = line.split()
        if len(fields) != 5:
            raise ValueError(f"{path}:{line_number}: se esperaban 5 columnas YOLO")
        try:
            class_id = int(fields[0])
            x, y, w, h = (float(v) for v in fields[1:])
        except ValueError as exc:
            raise ValueError(f"{path}:{line_number}: valor no numérico") from exc
        if not 0 <= class_id < len(PPE_CLASS_NAMES):
            raise ValueError(f"{path}:{line_number}: clase {class_id} fuera de rango")
        if not all(math.isfinite(v) for v in (x, y, w, h)):
            raise ValueError(f"{path}:{line_number}: coordenadas no finitas")
        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            raise ValueError(f"{path}:{line_number}: bbox normalizada inválida")
        counts[class_id] += 1
    return tuple(counts)


def infer_group(relative: Path, pattern: str | None) -> str:
    text = relative.as_posix()
    if pattern:
        match = re.search(pattern, text)
        if not match:
            raise ValueError(f"GROUP_REGEX no coincide con: {text}")
        return match.groupdict().get("group") or match.group(0)
    generic_containers = {"data", "dataset", "image", "images", "imagen", "imagenes"}
    if len(relative.parts) > 1 and relative.parts[0].lower() not in generic_containers:
        return relative.parts[0]
    stem = relative.stem
    # cam01_frame_000123 -> cam01; clipA-0012 -> clipA
    inferred = re.sub(r"(?i)(?:[_-]?(?:frame|img|image))?[_-]?\d+$", "", stem).rstrip("_-")
    return inferred if len(inferred) >= 3 else stem


def discover_pairs(images_root: Path, labels_root: Path, group_regex: str | None) -> list[Sample]:
    images = sorted(
        p for p in images_root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not images:
        raise FileNotFoundError(f"No hay imágenes en {images_root}")

    labels_by_stem: dict[str, list[Path]] = defaultdict(list)
    for label_path in sorted(labels_root.rglob("*.txt")):
        if label_path.is_file():
            labels_by_stem[label_path.stem].append(label_path)

    samples: list[Sample] = []
    seen_relatives: set[Path] = set()
    fallback_matches: list[tuple[Path, Path]] = []
    missing_labels: list[tuple[Path, Path]] = []
    for image in images:
        relative = image.relative_to(images_root)
        if relative in seen_relatives:
            raise ValueError(f"Imagen duplicada: {relative}")
        seen_relatives.add(relative)

        expected_label = labels_root / relative.with_suffix(".txt")
        if expected_label.is_file():
            label = expected_label
        else:
            candidates = labels_by_stem.get(image.stem, [])
            if len(candidates) == 1:
                label = candidates[0]
                fallback_matches.append((image, label))
            elif len(candidates) > 1:
                formatted = "\n  - ".join(str(path) for path in candidates)
                raise ValueError(
                    f"Etiqueta ambigua para {image}: hay {len(candidates)} archivos "
                    f"llamados {image.stem}.txt:\n  - {formatted}\n"
                    "Conserva la misma estructura relativa en images/ y labels/, "
                    "o renombra los archivos para que cada nombre sea único."
                )
            else:
                missing_labels.append((image, expected_label))
                continue

        samples.append(Sample(image, label, relative, parse_label(label), infer_group(relative, group_regex)))

    if missing_labels:
        preview = "\n".join(
            f"  - Imagen: {image}\n    Etiqueta esperada: {expected}"
            for image, expected in missing_labels[:20]
        )
        remainder = len(missing_labels) - 20
        extra = f"\n  ... y {remainder} caso(s) más." if remainder > 0 else ""
        raise FileNotFoundError(
            f"Faltan {len(missing_labels)} etiqueta(s). No se encontraron ni en la "
            "ruta espejo ni por nombre único dentro de labels/:\n"
            f"{preview}{extra}\n"
            "Si una imagen es realmente negativa, crea un .txt vacío con el mismo "
            "nombre. Si contiene EPP/personas, anótala; no generes la etiqueta vacía."
        )

    if fallback_matches:
        print(
            f"Aviso: {len(fallback_matches)} etiqueta(s) se emparejaron por nombre "
            "porque images/ y labels/ usan subcarpetas diferentes."
        )
    return samples


def class_totals(samples: Iterable[Sample]) -> np.ndarray:
    rows = [s.counts for s in samples]
    return np.asarray(rows, dtype=np.int64).sum(axis=0) if rows else np.zeros(len(PPE_CLASS_NAMES), dtype=np.int64)


def grouped_folds(samples: list[Sample], k: int) -> list[tuple[list[Sample], list[Sample]]]:
    if k < 2:
        raise ValueError("K_FOLDS debe ser al menos 2.")
    groups = [sample.group for sample in samples]
    distinct_groups = set(groups)
    if len(distinct_groups) < k:
        raise ValueError(
            f"Hay {len(distinct_groups)} secuencias para {k} folds. "
            "Reduce K_FOLDS o define GROUP_REGEX para identificar cámaras/secuencias."
        )
    if len(distinct_groups) == len(samples):
        print("ADVERTENCIA: cada imagen parece un grupo distinto. Revisa GROUP_REGEX si son frames vecinos.")

    folds = []
    validation_uses = np.zeros(len(samples), dtype=np.int64)
    for fold_number, (train_indices, val_indices) in enumerate(
        GroupKFold(n_splits=k).split(np.arange(len(samples)), groups=groups), start=1
    ):
        train = [samples[int(index)] for index in train_indices]
        val = [samples[int(index)] for index in val_indices]
        validation_uses[val_indices] += 1
        if {sample.group for sample in train} & {sample.group for sample in val}:
            raise RuntimeError(f"Fuga de secuencias en fold {fold_number}.")
        train_counts, val_counts = class_totals(train), class_totals(val)
        missing = [
            PPE_CLASS_NAMES[i] for i in range(len(PPE_CLASS_NAMES))
            if train_counts[i] == 0 or val_counts[i] == 0
        ]
        if missing:
            raise RuntimeError(
                f"Fold {fold_number} sin clases en train o val: {missing}. "
                "Reduce K_FOLDS o mejora la diversidad de secuencias por clase."
            )
        folds.append((train, val))
    if not np.all(validation_uses == 1):
        raise RuntimeError("Cada imagen debe validarse exactamente una vez en K-fold.")
    return folds


def link_or_copy(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(source, destination)
    except OSError:
        shutil.copy2(source, destination)


def reset_generated_dir(path: Path) -> None:
    marker = path / ".generated_by_epp_yolo26_cctv_v2"
    if path.exists():
        if not marker.is_file():
            raise RuntimeError(f"Se rechazó limpiar una carpeta no reconocida: {path}")
        shutil.rmtree(path)
    path.mkdir(parents=True)
    marker.write_text("generated\n", encoding="utf-8")


def materialize_split(split_root: Path, splits: dict[str, list[Sample]]) -> Path:
    reset_generated_dir(split_root)
    for split_name, samples in splits.items():
        for sample in samples:
            image_dst = split_root / "images" / split_name / sample.relative
            label_dst = split_root / "labels" / split_name / sample.relative.with_suffix(".txt")
            link_or_copy(sample.image, image_dst)
            link_or_copy(sample.label, label_dst)
    data = {
        "path": str(split_root.resolve()),
        "train": "images/train",
        "val": "images/val" if splits.get("val") else "images/train",
        "nc": len(PPE_CLASS_NAMES),
        "names": list(PPE_CLASS_NAMES),
    }
    if splits.get("test"):
        data["test"] = "images/test"
    yaml_path = split_root / "data.yaml"
    yaml_path.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return yaml_path


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def label_signature(path: Path) -> tuple[tuple[int | float, ...], ...]:
    """Representación semántica: ignora orden de filas y formato decimal."""
    rows: list[tuple[int | float, ...]] = []
    for raw in path.read_text(encoding="utf-8-sig").splitlines():
        fields = raw.split()
        if not fields:
            continue
        rows.append((int(fields[0]), *(round(float(value), 12) for value in fields[1:])))
    return tuple(sorted(rows))


def deduplicate_samples(samples: list[Sample], context: str) -> list[Sample]:
    """Elimina copias exactas equivalentes sin tocar los archivos fuente."""
    first_by_hash: dict[str, Sample] = {}
    unique: list[Sample] = []
    equivalent: list[tuple[Sample, Sample]] = []
    conflicts: list[tuple[Sample, Sample]] = []

    for sample in samples:
        image_hash = file_sha256(sample.image)
        previous = first_by_hash.get(image_hash)
        if previous is None:
            first_by_hash[image_hash] = sample
            unique.append(sample)
        elif label_signature(previous.label) == label_signature(sample.label):
            equivalent.append((previous, sample))
        else:
            conflicts.append((previous, sample))

    if conflicts:
        preview = "\n".join(
            f"  - Misma imagen:\n      {first.image}\n      {duplicate.image}\n"
            f"    Etiquetas diferentes:\n      {first.label}\n      {duplicate.label}"
            for first, duplicate in conflicts[:10]
        )
        raise RuntimeError(
            f"Hay {len(conflicts)} par(es) de imágenes idénticas con anotaciones "
            f"diferentes en {context}:\n{preview}\n"
            "Revisa y unifica esas anotaciones antes de entrenar."
        )

    if equivalent:
        print(
            f"Aviso: se excluyeron {len(equivalent)} copia(s) exacta(s) repetida(s) "
            f"de {context}; el dataset fuente no fue modificado."
        )
        for first, duplicate in equivalent[:5]:
            print(f"  Conservada: {first.image}\n  Excluida:   {duplicate.image}")
        if len(equivalent) > 5:
            print(f"  ... y {len(equivalent) - 5} copia(s) más.")
    return unique


def remove_cross_split_duplicates(
    splits: dict[str, list[Sample]],
) -> dict[str, list[Sample]]:
    """Da prioridad a train > val > test ante copias exactas equivalentes."""
    first_by_hash: dict[str, tuple[str, Sample]] = {}
    cleaned: dict[str, list[Sample]] = {name: [] for name in splits}
    equivalent: list[tuple[str, Sample, str, Sample]] = []
    conflicts: list[tuple[str, Sample, str, Sample]] = []

    for split_name in ("train", "val", "test"):
        for sample in splits.get(split_name, []):
            image_hash = file_sha256(sample.image)
            previous = first_by_hash.get(image_hash)
            if previous is None:
                first_by_hash[image_hash] = (split_name, sample)
                cleaned[split_name].append(sample)
                continue
            previous_split, previous_sample = previous
            row = (previous_split, previous_sample, split_name, sample)
            if label_signature(previous_sample.label) == label_signature(sample.label):
                equivalent.append(row)
            else:
                conflicts.append(row)

    if conflicts:
        preview = "\n".join(
            f"  - {first_split}: {first.image}\n"
            f"    {duplicate_split}: {duplicate.image}\n"
            f"    Etiquetas: {first.label} <> {duplicate.label}"
            for first_split, first, duplicate_split, duplicate in conflicts[:10]
        )
        raise RuntimeError(
            f"Hay {len(conflicts)} imagen(es) idéntica(s) entre splits con "
            f"anotaciones diferentes:\n{preview}\n"
            "No es seguro decidir automáticamente cuál anotación es correcta."
        )

    if equivalent:
        print(
            f"Aviso: se excluyeron {len(equivalent)} copia(s) idéntica(s) de splits "
            "posteriores para impedir fuga train/val/test. El dataset fuente sigue intacto."
        )
        for first_split, first, duplicate_split, duplicate in equivalent[:5]:
            print(
                f"  Conservada ({first_split}): {first.image}\n"
                f"  Excluida   ({duplicate_split}): {duplicate.image}"
            )
    return cleaned


def prepare_dataset(dataset_dir: Path) -> tuple[list[dict], list[Sample], list[Sample], Path]:
    if not dataset_dir.is_dir():
        raise FileNotFoundError(f"Dataset no encontrado: {dataset_dir}")
    load_dataset_names(dataset_dir)
    has_existing_split = all(
        (dataset_dir / kind / split).is_dir()
        for kind in ("images", "labels") for split in ("train", "val")
    )
    if has_existing_split:
        train = deduplicate_samples(
            discover_pairs(dataset_dir / "images/train", dataset_dir / "labels/train", GROUP_REGEX),
            "train",
        )
        val = deduplicate_samples(
            discover_pairs(dataset_dir / "images/val", dataset_dir / "labels/val", GROUP_REGEX),
            "val",
        )
        test = []
        if (dataset_dir / "images/test").is_dir() and (dataset_dir / "labels/test").is_dir():
            test = deduplicate_samples(
                discover_pairs(dataset_dir / "images/test", dataset_dir / "labels/test", GROUP_REGEX),
                "test",
            )
        cleaned = remove_cross_split_duplicates({"train": train, "val": val, "test": test})
        pool = [
            replace(sample, relative=Path(split) / sample.relative)
            for split in ("train", "val") for sample in cleaned[split]
        ]
        test = cleaned["test"]
    else:
        pool = deduplicate_samples(
            discover_pairs(dataset_dir / "images", dataset_dir / "labels", GROUP_REGEX),
            "dataset completo",
        )
        test = []

    if {sample.group for sample in pool} & {sample.group for sample in test}:
        raise RuntimeError("El test comparte secuencias con train/val; corrige GROUP_REGEX o el dataset.")
    folds = []
    manifest = []
    for number, (train, val) in enumerate(grouped_folds(pool, K_FOLDS), start=1):
        root = SPLIT_ROOT / f"fold_{number:02d}"
        data_yaml = materialize_split(root, {"train": train, "val": val})
        folds.append({"number": number, "root": root, "yaml": data_yaml, "train": train, "val": val})
        manifest.extend(
            {"fold": number, "image": str(sample.image), "group": sample.group}
            for sample in val
        )
    pd.DataFrame(manifest).to_csv(SPLIT_ROOT / "fold_manifest.csv", index=False)
    full_yaml = materialize_split(
        SPLIT_ROOT / "full_train", {"train": pool, "val": [], "test": test}
    )
    return folds, pool, test, full_yaml


FOLDS, POOL_SAMPLES, TEST_SAMPLES, FULL_DATA_YAML = prepare_dataset(DATASET_DIR)
print(f"GroupKFold: {K_FOLDS} folds, {len(POOL_SAMPLES)} imágenes, {len(TEST_SAMPLES)} test externo")
for fold in FOLDS:
    print(f"Fold {fold['number']}: {len(fold['train'])} train / {len(fold['val'])} val")
print("Asignaciones reproducibles:", SPLIT_ROOT / "fold_manifest.csv")


def available_ram_bytes() -> int:
    try:
        return int(os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"))
    except (AttributeError, OSError, ValueError):
        return 0


def resolve_cache_mode(mode: str | bool, samples: list[Sample], imgsz: int) -> str | bool:
    if mode != "auto":
        return mode
    # Aproximación conservadora del caché de imágenes decodificadas y redimensionadas.
    estimated = int(len(samples) * imgsz * imgsz * 3 * 1.5)
    available = available_ram_bytes()
    selected = "ram" if available and estimated <= available * 0.30 else "disk"
    print(
        f"Caché automático: {selected} "
        f"(estimado {estimated / 2**30:.2f} GiB; RAM disponible {available / 2**30:.2f} GiB)"
    )
    return selected


TRAIN_CACHE = resolve_cache_mode(CACHE_MODE, POOL_SAMPLES, IMGSZ)

rows = []
for fold in FOLDS:
    train_counts, val_counts = class_totals(fold["train"]), class_totals(fold["val"])
    for class_id, class_name in enumerate(PPE_CLASS_NAMES):
        rows.append({
            "fold": fold["number"],
            "id": class_id,
            "class": class_name,
            "train_images": sum(s.counts[class_id] > 0 for s in fold["train"]),
            "train_instances": int(train_counts[class_id]),
            "val_images": sum(s.counts[class_id] > 0 for s in fold["val"]),
            "val_instances": int(val_counts[class_id]),
        })
support_df = pd.DataFrame(rows)
display(support_df)
if (support_df[["train_instances", "val_instances"]] == 0).any().any():
    raise RuntimeError("Al menos una clase quedó sin soporte en un fold; revisa K_FOLDS o GROUP_REGEX.")



## 4. Validaciones limpias e inmutables por fold

Se congela **antes** de crear cualquier degradación. Si ya existe, sus hashes
deben coincidir; nunca se sobrescribe silenciosamente.



In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def build_manifest(images_root: Path, labels_root: Path) -> list[dict[str, str | int]]:
    rows = []
    for image in sorted(p for p in images_root.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS):
        relative = image.relative_to(images_root)
        label = labels_root / relative.with_suffix(".txt")
        rows.append({
            "image": relative.as_posix(),
            "label": relative.with_suffix(".txt").as_posix(),
            "image_sha256": sha256(image),
            "label_sha256": sha256(label),
            "instances": len([x for x in label.read_text(encoding="utf-8-sig").splitlines() if x.strip()]),
        })
    return rows


def freeze_benchmark(source_root: Path, benchmark_root: Path) -> Path:
    source_images = source_root / "images/val"
    source_labels = source_root / "labels/val"
    current_manifest = build_manifest(source_images, source_labels)
    manifest_path = benchmark_root / "manifest.json"
    if benchmark_root.exists():
        if not manifest_path.is_file():
            raise RuntimeError(f"Benchmark existente sin manifest: {benchmark_root}")
        previous = json.loads(manifest_path.read_text(encoding="utf-8"))
        if previous != current_manifest:
            raise RuntimeError(
                "El benchmark congelado no coincide con el split actual. Usa otro BENCHMARK_ROOT; no lo sobrescribas."
            )
    else:
        benchmark_root.mkdir(parents=True)
        shutil.copytree(source_images, benchmark_root / "images/val")
        shutil.copytree(source_labels, benchmark_root / "labels/val")
        manifest_path.write_text(json.dumps(current_manifest, indent=2), encoding="utf-8")
        benchmark_yaml = {
            "path": str(benchmark_root.resolve()),
            "train": "images/val",
            "val": "images/val",
            "test": "images/val",
            "nc": len(PPE_CLASS_NAMES),
            "names": list(PPE_CLASS_NAMES),
        }
        (benchmark_root / "data.yaml").write_text(
            yaml.safe_dump(benchmark_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8"
        )
    return benchmark_root / "data.yaml"


BENCHMARK_YAMLS = {
    fold["number"]: freeze_benchmark(
        fold["root"], BENCHMARK_ROOT / f"fold_{fold['number']:02d}"
    )
    for fold in FOLDS
}
for fold in FOLDS:
    print(f"Fold {fold['number']}: {len(fold['val'])} imágenes congeladas en {BENCHMARK_YAMLS[fold['number']]}")



## 5. Aumentación CCTV/RTSP moderada

El protocolo RTSP no altera píxeles por sí mismo; los artefactos provienen del
códec, bitrate, pérdida/recuperación de paquetes, exposición, sensor, óptica y
movimiento. Durante entrenamiento usamos aproximaciones que no añaden ninguna
latencia a producción:

- compresión JPEG y reducción/reescalado como proxy de bajo bitrate;
- ruido, gamma y contraste para sensor/iluminación;
- blur de movimiento de baja probabilidad;
- niebla ligera y pequeños parches para atmósfera/oclusión.

No se apilan degradaciones extremas. Turbulencia geométrica fuerte, pérdida de
bloques H.264 y cámaras congeladas deben evaluarse en un benchmark de video,
porque una transformación por imagen puede volver incorrectas las cajas.



In [ ]:
def build_cctv_augmentations() -> list[A.BasicTransform]:
    return [
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=(-0.30, 0.15), contrast_limit=(-0.20, 0.25), p=1.0
            ),
            A.RandomGamma(gamma_limit=(65, 145), p=1.0),
        ], p=0.30),
        A.OneOf([
            A.ImageCompression(quality_range=(45, 92), p=1.0),
            A.Downscale(scale_range=(0.55, 0.90), p=1.0),
            A.GaussNoise(std_range=(0.005, 0.030), p=1.0),
        ], p=0.25),
        A.OneOf([
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        ], p=MOTION_BLUR_P),
        A.RandomFog(fog_coef_range=(0.05, 0.18), alpha_coef=0.08, p=ATMOSPHERIC_P),
        A.ToGray(p=0.02),
        A.CoarseDropout(
            num_holes_range=(1, 3),
            hole_height_range=(0.02, 0.07),
            hole_width_range=(0.02, 0.07),
            fill=0,
            p=OCCLUSION_P,
        ),
    ]


CCTV_AUGMENTATIONS = build_cctv_augmentations()
print(A.Compose(CCTV_AUGMENTATIONS))


def read_yolo_boxes(label: Path) -> tuple[list[list[float]], list[int]]:
    boxes, classes = [], []
    for line in label.read_text(encoding="utf-8-sig").splitlines():
        if line.strip():
            values = line.split()
            classes.append(int(values[0]))
            boxes.append([float(v) for v in values[1:]])
    return boxes, classes


def draw_yolo(image_rgb: np.ndarray, boxes: list[list[float]], classes: list[int]) -> np.ndarray:
    output = image_rgb.copy()
    height, width = output.shape[:2]
    for (xc, yc, bw, bh), class_id in zip(boxes, classes):
        x1, y1 = int((xc - bw / 2) * width), int((yc - bh / 2) * height)
        x2, y2 = int((xc + bw / 2) * width), int((yc + bh / 2) * height)
        cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(output, PPE_CLASS_NAMES[class_id], (x1, max(16, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)
    return output


preview_transform = A.Compose(
    CCTV_AUGMENTATIONS,
    bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.10),
)
preview_train = FOLDS[0]["train"]
preview_samples = random.Random(SEED).sample(preview_train, k=min(6, len(preview_train)))
fig, axes = plt.subplots(len(preview_samples), 2, figsize=(14, 5 * len(preview_samples)))
axes = np.atleast_2d(axes)
for row, sample in enumerate(preview_samples):
    image = cv2.cvtColor(cv2.imread(str(sample.image)), cv2.COLOR_BGR2RGB)
    boxes, classes = read_yolo_boxes(sample.label)
    augmented = preview_transform(image=image, bboxes=boxes, class_labels=classes)
    axes[row, 0].imshow(draw_yolo(image, boxes, classes))
    axes[row, 0].set_title("Original")
    axes[row, 1].imshow(draw_yolo(augmented["image"], augmented["bboxes"], augmented["class_labels"]))
    axes[row, 1].set_title("CCTV/RTSP sintético")
    for axis in axes[row]:
        axis.axis("off")
plt.tight_layout()
plt.show()



## 6. Entrenamiento K-fold y ajuste final FP16

- FP16 aquí es precisión mixta de entrenamiento (`amp="fp16"`), no INT8.
- AutoBatch busca el lote más grande dentro del 75 % de la VRAM.
- El caché usa RAM solo si cabe con margen; si no, usa el SSD local de Colab.
- `torch.compile` y `channels_last` reducen overhead y favorecen Tensor Cores.
- Se mantiene tamaño fijo en 960 para evitar recompilaciones por `multi_scale`.
- `cls_pw=0.25` pondera parcialmente clases escasas; compáralo en una ablación
  contra `0.0` si suben los falsos positivos.
- `mosaic`, `cutmix` y transformaciones geométricas son deliberadamente bajos.
- Cada fold empieza desde `BASE_MODEL` y selecciona su checkpoint con la
  misma cabeza que producción. Se entrenan K modelos independientes.
- Después se ajusta un modelo final con todo train+val durante la mediana
  de las mejores épocas de los folds, sin usar test ni sus métricas para elegirlo.
  Con `K_FOLDS=5` hay cinco entrenamientos de validación y un ajuste final.
  Ultralytics puede calcular métricas sobre train al cerrar ese ajuste; no
  representan validación externa. Usa las métricas K-fold o el test reservado.



In [ ]:
TRAIN_ARGS = {
    "data": str(FOLDS[0]["yaml"]),
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "device": DEVICE,
    "workers": WORKERS,
    "cache": TRAIN_CACHE,
    "seed": SEED,
    "deterministic": not FAST_TRAINING,
    "amp": "fp16",
    "compile": COMPILE_MODE,
    "channels_last": True,
    "optimizer": "auto",
    "cos_lr": True,
    "save_period": 10,
    "project": str(RUNS_ROOT),
    "name": RUN_NAME + "_fold_01",
    "exist_ok": False,
    "plots": True,
    "nms": PRODUCTION_NMS,
    "cls_pw": CLS_PW,
    "hsv_h": 0.010,
    "hsv_s": 0.25,
    "hsv_v": 0.22,
    "degrees": 1.5,
    "translate": 0.06,
    "scale": 0.30,
    "shear": 0.0,
    "perspective": 0.0003,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 0.35,
    "close_mosaic": 15,
    "mixup": 0.0,
    "cutmix": 0.02,
    "copy_paste": 0.0,
    "multi_scale": 0.0,
    "augmentations": CCTV_AUGMENTATIONS,
}

if FAST_TRAINING:
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

CV_RUNS = []
for fold in FOLDS:
    fold_number = fold["number"]
    fold_args = {**TRAIN_ARGS, "data": str(fold["yaml"]), "name": f"{RUN_NAME}_fold_{fold_number:02d}"}
    print(f"Entrenando fold {fold_number}/{K_FOLDS}: {fold['yaml']}")
    fold_model = YOLO(BASE_MODEL)
    fold_model.train(**fold_args)
    fold_run_dir = Path(fold_model.trainer.save_dir)
    fold_best_pt = fold_run_dir / "weights/best.pt"
    if not fold_best_pt.is_file():
        raise FileNotFoundError(f"No se generó best.pt para fold {fold_number}")
    history = pd.read_csv(fold_run_dir / "results.csv")
    history.columns = history.columns.str.strip()
    metric_column = "metrics/mAP50-95(B)"
    best_epoch = int(history[metric_column].astype(float).idxmax()) + 1 if metric_column in history else len(history)
    CV_RUNS.append({"number": fold_number, "run_dir": fold_run_dir, "best_pt": fold_best_pt,
                    "best_epoch": best_epoch, "benchmark_yaml": BENCHMARK_YAMLS[fold_number]})
    shutil.copytree(fold_run_dir, DRIVE_RUNS_ROOT / fold_run_dir.name, dirs_exist_ok=True)
    print(f"Fold {fold_number}: mejor época {best_epoch}, checkpoint {fold_best_pt}")
    del fold_model
    torch.cuda.empty_cache()

FINAL_EPOCHS = max(1, round(float(np.median([run["best_epoch"] for run in CV_RUNS]))))
FINAL_TRAIN_ARGS = {**TRAIN_ARGS, "data": str(FULL_DATA_YAML), "name": RUN_NAME + "_full_train",
                    "epochs": FINAL_EPOCHS, "patience": FINAL_EPOCHS + 1, "val": False}
print(f"Ajuste final con {len(POOL_SAMPLES)} imágenes durante {FINAL_EPOCHS} épocas")
final_model = YOLO(BASE_MODEL)
final_model.train(**FINAL_TRAIN_ARGS)
RUN_DIR = Path(final_model.trainer.save_dir)
FINAL_PT = RUN_DIR / "weights/last.pt"
if not FINAL_PT.is_file():
    raise FileNotFoundError(f"No se generó last.pt en el ajuste final: {RUN_DIR}")
shutil.copy2(FINAL_PT, OUTPUT_MODEL)
del final_model
torch.cuda.empty_cache()
with (RUN_DIR / "environment.txt").open("w", encoding="utf-8") as environment_file:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, stdout=environment_file)
(RUN_DIR / "experiment.json").write_text(
    json.dumps({
        "dataset": str(DATASET_DIR),
        "split_method": "GroupKFold",
        "k_folds": K_FOLDS,
        "folds": [{"number": run["number"], "run_dir": str(run["run_dir"]),
                   "best_epoch": run["best_epoch"]} for run in CV_RUNS],
        "final_epochs": FINAL_EPOCHS,
        "benchmark": str(BENCHMARK_ROOT),
        "classes": list(PPE_CLASS_NAMES),
        "production_nms": PRODUCTION_NMS,
        "train_args": {k: repr(v) if k == "augmentations" else v for k, v in FINAL_TRAIN_ARGS.items()},
        "versions": {
            "python": sys.version,
            "torch": torch.__version__,
            "ultralytics": ultralytics.__version__,
            "albumentations": A.__version__,
        },
    }, indent=2, default=str), encoding="utf-8"
)
DRIVE_RUN_DIR = DRIVE_RUNS_ROOT / RUN_DIR.name
shutil.copytree(RUN_DIR, DRIVE_RUN_DIR, dirs_exist_ok=True)
print("Modelo final (ajustado con todo train+val):", OUTPUT_MODEL)
print("Run local:", RUN_DIR)
print("Run respaldado:", DRIVE_RUN_DIR)



## 7. Evaluación limpia en todos los folds

El umbral `conf=0.001` es intencional para calcular curvas PR completas. No es
el umbral de alarma de producción. Los gráficos `PR_curve.png`, `F1_curve.png`
y la matriz de confusión sirven luego para escoger umbrales por clase.
Las medias y desviaciones se calculan sobre los K conjuntos de validación;
el modelo final entrenado con todo train+val solo se evalúa en `test` si existe.



In [ ]:
def metric_value(metrics, key: str) -> float:
    value = metrics.results_dict.get(key, float("nan"))
    return float(value.cpu()) if hasattr(value, "cpu") else float(value)


def per_class_metrics(metrics, evaluation: str, uses_nms: bool) -> pd.DataFrame:
    def values(attribute: str) -> np.ndarray:
        value = getattr(metrics.box, attribute, np.full(len(PPE_CLASS_NAMES), np.nan))
        value = value.cpu().numpy() if hasattr(value, "cpu") else np.asarray(value)
        return value.astype(float)

    p, r, f1, ap50, maps = (values(x) for x in ("p", "r", "f1", "ap50", "maps"))
    rows = []
    for i, name in enumerate(PPE_CLASS_NAMES):
        rows.append({
            "evaluation": evaluation,
            "nms": uses_nms,
            "class_id": i,
            "class": name,
            "precision": p[i] if i < len(p) else np.nan,
            "recall": r[i] if i < len(r) else np.nan,
            "f1": f1[i] if i < len(f1) else np.nan,
            "ap50": ap50[i] if i < len(ap50) else np.nan,
            "map50_95": maps[i] if i < len(maps) else np.nan,
        })
    return pd.DataFrame(rows)


production_evaluation = "production_end2end_nms_false" if not PRODUCTION_NMS else "production_nms"
head_configs = {production_evaluation: PRODUCTION_NMS}
if COMPARE_BOTH_HEADS:
    alternative = "control_one_to_many_nms" if not PRODUCTION_NMS else "alternative_end2end"
    head_configs[alternative] = not PRODUCTION_NMS

head_tables = []
head_summary = []
for run in CV_RUNS:
    fold_model = YOLO(str(run["best_pt"]))
    for evaluation, uses_nms in head_configs.items():
        metrics = fold_model.val(
            data=str(run["benchmark_yaml"]),
            split="val",
            imgsz=IMGSZ,
            batch=VAL_BATCH,
            device=DEVICE,
            conf=0.001,
            iou=0.70,
            nms=uses_nms,
            plots=True,
            project=str(run["run_dir"] / "evaluation"),
            name=evaluation,
        )
        class_table = per_class_metrics(metrics, evaluation, uses_nms)
        class_table.insert(0, "fold", run["number"])
        head_tables.append(class_table)
        head_summary.append({
            "fold": run["number"], "evaluation": evaluation, "nms": uses_nms,
            "precision": metric_value(metrics, "metrics/precision(B)"),
            "recall": metric_value(metrics, "metrics/recall(B)"),
            "map50": metric_value(metrics, "metrics/mAP50(B)"),
            "map50_95": metric_value(metrics, "metrics/mAP50-95(B)"),
        })
    del fold_model
    torch.cuda.empty_cache()

head_summary_df = pd.DataFrame(head_summary)
per_class_df = pd.concat(head_tables, ignore_index=True)
metric_columns = ("precision", "recall", "map50", "map50_95")
head_aggregate = []
for evaluation, rows in head_summary_df.groupby("evaluation"):
    summary = {"evaluation": evaluation, "nms": bool(rows["nms"].iloc[0])}
    for column in metric_columns:
        summary[f"{column}_mean"] = rows[column].mean()
        summary[f"{column}_std"] = rows[column].std(ddof=1)
    head_aggregate.append(summary)
head_aggregate_df = pd.DataFrame(head_aggregate)
class_aggregate = []
for (evaluation, class_id, name), rows in per_class_df.groupby(["evaluation", "class_id", "class"]):
    summary = {"evaluation": evaluation, "class_id": class_id, "class": name}
    for column in ("precision", "recall", "f1", "ap50", "map50_95"):
        summary[f"{column}_mean"] = rows[column].mean()
        summary[f"{column}_std"] = rows[column].std(ddof=1)
    class_aggregate.append(summary)
class_aggregate_df = pd.DataFrame(class_aggregate)
CV_REPORT_ROOT = DRIVE_RUNS_ROOT / f"{RUN_NAME}_cv_reports"
CV_REPORT_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(SPLIT_ROOT / "fold_manifest.csv", CV_REPORT_ROOT / "fold_manifest.csv")
support_df.to_csv(CV_REPORT_ROOT / "class_support_by_fold.csv", index=False)
head_summary_df.to_csv(CV_REPORT_ROOT / "head_summary_by_fold.csv", index=False)
head_aggregate_df.to_csv(CV_REPORT_ROOT / "head_summary_mean_std.csv", index=False)
per_class_df.to_csv(CV_REPORT_ROOT / "per_class_by_fold.csv", index=False)
class_aggregate_df.to_csv(CV_REPORT_ROOT / "per_class_mean_std.csv", index=False)
head_summary_df.to_csv(RUN_DIR / "head_summary.csv", index=False)
per_class_df.to_csv(RUN_DIR / "per_class_metrics.csv", index=False)
display(head_aggregate_df)
display(class_aggregate_df)
if TEST_SAMPLES:
    test_metrics = YOLO(str(FINAL_PT)).val(
        data=str(FULL_DATA_YAML), split="test", imgsz=IMGSZ, batch=VAL_BATCH,
        device=DEVICE, conf=0.001, iou=0.70, nms=PRODUCTION_NMS,
        plots=True, project=str(RUN_DIR / "evaluation"), name="heldout_test",
    )
    test_summary = {column: metric_value(test_metrics, key) for column, key in {
        "precision": "metrics/precision(B)", "recall": "metrics/recall(B)",
        "map50": "metrics/mAP50(B)", "map50_95": "metrics/mAP50-95(B)",
    }.items()}
    pd.DataFrame([test_summary]).to_csv(CV_REPORT_ROOT / "heldout_test_summary.csv", index=False)
    print("Test externo del modelo final:", test_summary)
shutil.copytree(RUN_DIR, DRIVE_RUN_DIR, dirs_exist_ok=True)
for run in CV_RUNS:
    shutil.copytree(run["run_dir"], DRIVE_RUNS_ROOT / run["run_dir"].name, dirs_exist_ok=True)



## 8. Benchmark de degradaciones por fold (solo evaluación)

Esto no agrega latencia a inferencia. Crea copias degradadas del conjunto
congelado y cuantifica cuánto cae cada métrica. Las transformaciones conservan
geometría, por lo que las cajas siguen siendo válidas.



In [ ]:
def degrade_image(image: np.ndarray, profile: str, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    if profile == "low_light_noise":
        normalized = np.clip(image.astype(np.float32) / 255.0, 0, 1)
        dark = np.power(normalized, 1.8) * 255.0
        noisy = dark + rng.normal(0, 7.0, image.shape)
        return np.clip(noisy, 0, 255).astype(np.uint8)
    if profile == "compression_q35":
        ok, encoded = cv2.imencode(".jpg", image, [cv2.IMWRITE_JPEG_QUALITY, 35])
        if not ok:
            raise RuntimeError("No se pudo codificar JPEG")
        return cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    if profile == "downscale_050":
        h, w = image.shape[:2]
        small = cv2.resize(image, (max(1, w // 2), max(1, h // 2)), interpolation=cv2.INTER_AREA)
        return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    if profile == "motion_blur_7":
        kernel = np.zeros((7, 7), dtype=np.float32)
        kernel[3, :] = 1.0 / 7.0
        return cv2.filter2D(image, -1, kernel)
    if profile == "fog_haze":
        haze = np.full_like(image, 205)
        return cv2.addWeighted(image, 0.72, haze, 0.28, 0)
    raise KeyError(profile)


def create_degraded_dataset(profile: str, fold_number: int) -> Path:
    root = WORK_ROOT / "robustness" / f"fold_{fold_number:02d}" / profile
    marker = root / ".generated_by_epp_yolo26_cctv_v2"
    if root.exists():
        if not marker.is_file():
            raise RuntimeError(f"Se rechazó limpiar una carpeta no reconocida: {root}")
        shutil.rmtree(root)
    (root / "images/val").mkdir(parents=True)
    (root / "labels/val").mkdir(parents=True)
    marker.write_text("generated\n", encoding="utf-8")

    benchmark_fold_root = BENCHMARK_ROOT / f"fold_{fold_number:02d}"
    source_images = benchmark_fold_root / "images/val"
    source_labels = benchmark_fold_root / "labels/val"
    for index, image_path in enumerate(sorted(p for p in source_images.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)):
        relative = image_path.relative_to(source_images)
        label_path = source_labels / relative.with_suffix(".txt")
        output_image = (root / "images/val" / relative).with_suffix(".jpg")
        output_label = root / "labels/val" / relative.with_suffix(".txt")
        output_image.parent.mkdir(parents=True, exist_ok=True)
        output_label.parent.mkdir(parents=True, exist_ok=True)
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise RuntimeError(f"OpenCV no pudo leer {image_path}")
        degraded = degrade_image(image, profile, SEED + fold_number * 1_000_000 + index)
        if not cv2.imwrite(str(output_image), degraded, [cv2.IMWRITE_JPEG_QUALITY, 95]):
            raise RuntimeError(f"OpenCV no pudo escribir {output_image}")
        shutil.copy2(label_path, output_label)

    data = {
        "path": str(root.resolve()),
        "train": "images/val",
        "val": "images/val",
        "nc": len(PPE_CLASS_NAMES),
        "names": list(PPE_CLASS_NAMES),
    }
    data_yaml = root / "data.yaml"
    data_yaml.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return data_yaml


if RUN_ROBUSTNESS_BENCHMARK:
    robustness_rows = []
    for run in CV_RUNS:
        fold_model = YOLO(str(run["best_pt"]))
        for profile in ("low_light_noise", "compression_q35", "downscale_050", "motion_blur_7", "fog_haze"):
            profile_yaml = create_degraded_dataset(profile, run["number"])
            metrics = fold_model.val(
                data=str(profile_yaml),
                split="val",
                imgsz=IMGSZ,
                batch=VAL_BATCH,
                device=DEVICE,
                conf=0.001,
                iou=0.70,
                nms=PRODUCTION_NMS,
                plots=False,
                project=str(run["run_dir"] / "robustness"),
                name=profile,
            )
            robustness_rows.append({
                "fold": run["number"], "profile": profile,
                "precision": metric_value(metrics, "metrics/precision(B)"),
                "recall": metric_value(metrics, "metrics/recall(B)"),
                "map50": metric_value(metrics, "metrics/mAP50(B)"),
                "map50_95": metric_value(metrics, "metrics/mAP50-95(B)"),
            })
        del fold_model
        torch.cuda.empty_cache()
        shutil.copytree(run["run_dir"], DRIVE_RUNS_ROOT / run["run_dir"].name, dirs_exist_ok=True)
    robustness_df = pd.DataFrame(robustness_rows)
    robustness_df.to_csv(CV_REPORT_ROOT / "robustness_by_fold.csv", index=False)
    robustness_aggregate = []
    for profile, rows in robustness_df.groupby("profile"):
        summary = {"profile": profile}
        for column in metric_columns:
            summary[f"{column}_mean"] = rows[column].mean()
            summary[f"{column}_std"] = rows[column].std(ddof=1)
        robustness_aggregate.append(summary)
    robustness_aggregate_df = pd.DataFrame(robustness_aggregate)
    robustness_aggregate_df.to_csv(CV_REPORT_ROOT / "robustness_mean_std.csv", index=False)
    robustness_aggregate_df.to_csv(RUN_DIR / "robustness_summary.csv", index=False)
    shutil.copytree(RUN_DIR, DRIVE_RUN_DIR, dirs_exist_ok=True)
    display(robustness_aggregate_df)



## 9. QAT INT8 opcional

Ejecuta esta celda solo después de construir un engine INT8 PTQ en el GB10 y
comprobar que la caída frente a FP16 excede el criterio acordado (por ejemplo,
recall por clase o F1 de `Gloves`/`lentes_protectores`). QAT simula INT8 durante
un fine-tuning corto; no sustituye la evaluación del engine real.



In [ ]:
if RUN_QAT_INT8:
    qat_model = YOLO(str(FINAL_PT))
    qat_results = qat_model.train(
        data=str(FULL_DATA_YAML),
        epochs=QAT_EPOCHS,
        patience=QAT_EPOCHS + 1,
        val=False,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        cache=TRAIN_CACHE,
        seed=SEED,
        deterministic=True,
        amp="fp16",
        quantize=8,
        optimizer="AdamW",
        lr0=1e-4,
        lrf=0.1,
        mosaic=0.0,
        mixup=0.0,
        cutmix=0.0,
        multi_scale=0.0,
        augmentations=CCTV_AUGMENTATIONS,
        nms=PRODUCTION_NMS,
        cls_pw=CLS_PW,
        project=str(RUNS_ROOT),
        name=RUN_NAME + "_qat_int8",
        plots=True,
    )
    qat_run_dir = Path(qat_model.trainer.save_dir)
    qat_best = qat_run_dir / "weights/last.pt"
    qat_output = OUTPUT_MODEL.with_name(OUTPUT_MODEL.stem + "_qat_int8.pt")
    shutil.copy2(qat_best, qat_output)
    shutil.copytree(qat_run_dir, DRIVE_RUNS_ROOT / qat_run_dir.name, dirs_exist_ok=True)
    print("Modelo QAT:", qat_output)
else:
    print("QAT INT8 desactivado. Correcto para el primer baseline.")



## 10. Exportación ONNX end-to-end sin NMS

`nms=False` selecciona la rama one-to-one. `nms=None` **no** significa lo mismo:
produciría la salida raw one-to-many que requiere NMS externo. Se exporta ONNX
FP32 para conservar compatibilidad; la precisión FP16/INT8 del engine TensorRT
se decide y valida en el GB10 objetivo.



In [ ]:
ONNX_OUTPUT = OUTPUT_MODEL.with_name(OUTPUT_MODEL.stem + "_e2e_nmsfalse_fp32.onnx")
if EXPORT_END2END_ONNX:
    export_model = YOLO(str(FINAL_PT))
    exported_path = Path(export_model.export(
        format="onnx",
        imgsz=IMGSZ,
        batch=1,
        dynamic=False,
        simplify=True,
        quantize=32,
        nms=False,
        max_det=300,
        device=DEVICE,
    ))
    graph = onnx.load(str(exported_path))
    output_dims = []
    for dim in graph.graph.output[0].type.tensor_type.shape.dim:
        output_dims.append(dim.dim_value if dim.HasField("dim_value") else dim.dim_param)
    print("Shape ONNX:", output_dims)
    if output_dims != [1, 300, 6]:
        raise RuntimeError(
            f"Se esperaba [1, 300, 6] para nms=False, pero se obtuvo {output_dims}. "
            "No despliegues este artefacto."
        )
    shutil.copy2(exported_path, ONNX_OUTPUT)
    print("ONNX end-to-end:", ONNX_OUTPUT)
else:
    print("Exportación ONNX desactivada.")



## 11. Cierre y criterios de aceptación

Antes de promover el modelo:

1. Compara las medias y desviaciones K-fold de `head_summary_mean_std.csv` y
   `per_class_mean_std.csv` con el modelo anterior.
2. Revisa en particular recall y falsos negativos de `Gloves`, `respirador` y
   `lentes_protectores`.
3. Define límites de degradación aceptables con `robustness_mean_std.csv`.
4. Elige umbrales por clase con las curvas PR/F1, no un `conf=0.25` universal.
5. En producción exige persistencia temporal y tracking antes de una alarma;
   para una ausencia de EPP, usa varias observaciones consecutivas.
6. Construye TensorRT FP16/INT8 en el GB10 objetivo y repite el benchmark completo.
7. Verifica que el decoder C++ reciba `[1,300,6]` y no ejecute NMS.



In [ ]:
# Sincronización final después de evaluaciones.
shutil.copytree(RUN_DIR, DRIVE_RUN_DIR, dirs_exist_ok=True)
print("Trabajo completado")
print("Modelo:", OUTPUT_MODEL)
print("Resultados del modelo final:", DRIVE_RUN_DIR)
print("Resumen K-fold:", CV_REPORT_ROOT)
print("Validaciones limpias por fold:", BENCHMARK_ROOT)
